# Download the dataset for publication

Exports the Icechunk store to a self-contained plain Zarr v3 store and archives it as a single `.zip` for upload to Zenodo.

1. **Copy** every Zarr key byte-for-byte out of Icechunk (no decode/re-encode, so shards and compression are preserved exactly)
2. **Patch** the root/variable attributes with publication metadata
3. **Verify** against the source store
4. **Archive** as `ZIP_STORED` — chunks are already compressed, and the zip stays readable without extracting

In [1]:
import asyncio
import hashlib
import json
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import xarray as xr
import zarr
from tqdm.auto import tqdm

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))
from modis_snow_phenology import Config

config = Config('config/config_with_secrets_v1.txt')

OUT_DIR = REPO_ROOT / 'dataset'
STORE_PATH = OUT_DIR / f'modis_snow_phenology_{config.VERSION}.zarr'
ZIP_PATH = OUT_DIR / f'modis_snow_phenology_{config.VERSION}.zarr.zip'

ZENODO_DOI = "10.5281/zenodo.21783366"

CONCURRENCY = 32
BUFFER_PROTOTYPE = zarr.core.buffer.default_buffer_prototype()

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'store : {STORE_PATH}')
print(f'zip   : {ZIP_PATH}')

store : /home/eric/repos/MODIS_snow_phenology/dataset/modis_snow_phenology_v1.zarr
zip   : /home/eric/repos/MODIS_snow_phenology/dataset/modis_snow_phenology_v1.zarr.zip


In [2]:
# Open the source store, pinned to the current snapshot so the export is reproducible
repo = config.open_icechunk_repo()
session = repo.readonly_session('main')
src = session.store
snapshot_id = session.snapshot_id

ds_src = xr.open_zarr(src, zarr_format=3, consolidated=False, decode_coords='all')
print(f'snapshot: {snapshot_id}')
ds_src

snapshot: M64XYVMFDN7GN756C3AG


<xarray.Dataset> Size: 493GB
Dimensions:               (water_year: 11, y: 43200, x: 86400)
Coordinates:
  * water_year            (water_year) int64 88B 2015 2016 2017 ... 2024 2025
  * y                     (y) float64 346kB 1.001e+07 1.001e+07 ... -1.001e+07
  * x                     (x) float64 691kB -2.001e+07 -2.001e+07 ... 2.001e+07
    spatial_ref           int64 8B ...
Data variables:
    SAD_DOWY              (water_year, y, x) float32 164GB dask.array<chunksize=(1, 600, 600), meta=np.ndarray>
    max_consec_snow_days  (water_year, y, x) float32 164GB dask.array<chunksize=(1, 600, 600), meta=np.ndarray>
    SDD_DOWY              (water_year, y, x) float32 164GB dask.array<chunksize=(1, 600, 600), meta=np.ndarray>
Attributes:
    title:        Global MODIS Snow Phenology
    description:  Snow appearance date (SAD), snow disappearance date (SDD), ...
    source:       MODIS MOD10A2.061 via Microsoft Planetary Computer
    Conventions:  CF-1.8

## 1. Copy the store

Keys already present locally are skipped, so an interrupted run can just be re-run.

In [3]:
async def copy_store(src_store, dst_path, concurrency=CONCURRENCY):
    """Copy every key from src_store to a plain Zarr store at dst_path, verbatim."""
    dst_store = zarr.storage.LocalStore(dst_path)
    keys = [k async for k in src_store.list()]
    sem = asyncio.Semaphore(concurrency)
    pbar = tqdm(total=len(keys), desc='copying keys', unit='key')

    async def copy_one(key):
        async with sem:
            out = Path(dst_path) / key
            if not (out.exists() and out.stat().st_size > 0):
                buf = await src_store.get(key, BUFFER_PROTOTYPE)
                if buf is None:
                    raise KeyError(f'source returned nothing for {key}')
                await dst_store.set(key, buf)
            pbar.update(1)

    await asyncio.gather(*(copy_one(k) for k in keys))
    pbar.close()
    return keys


src_keys = await copy_store(src, STORE_PATH)

local_files = [p for p in STORE_PATH.rglob('*') if p.is_file()]
store_bytes = sum(p.stat().st_size for p in local_files)
print(f'{len(local_files):,} files, {store_bytes / 1e9:.2f} GB')

9,979 files, 3.60 GB


## 2. Add publication metadata

Metadata-only edit of `zarr.json` — no data is touched. Also corrects the stale `source` attribute (the pipeline moved from Planetary Computer to NSIDC/earthaccess).

In [4]:
created = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

root_attrs = {
    'title': 'Global MODIS snow phenology (snow appearance date, snow disappearance date, '
             'maximum consecutive snow days), water years 2015-2025',
    'summary': 'Annual global maps of three snow phenology metrics at 500 m resolution, derived from the '
               'MODIS/Terra MOD10A2 Version 61 8-day maximum snow extent product for water years '
               f'{config.WY_START}-{config.WY_END}. Snow appearance date (SAD_DOWY) and snow disappearance '
               'date (SDD_DOWY) bracket the longest continuous snow-covered period of each water year, whose '
               'length is max_consec_snow_days. Dates are given as day of water year (DOWY), where DOWY 1 is '
               'Oct 1 in the northern hemisphere and Apr 1 in the southern hemisphere.',
    'keywords': 'snow, snow cover, snow phenology, snow appearance date, snow disappearance date, '
                'snow duration, MODIS, MOD10A2, cryosphere, hydrology, remote sensing, zarr',
    'Conventions': 'CF-1.8, ACDD-1.3',
    'product_version': '1.0.0',
    'source': 'MODIS/Terra Snow Cover 8-Day L3 Global 500m SIN Grid, Version 61 (MOD10A2.061), '
              'NASA National Snow and Ice Data Center DAAC',
    'source_doi': 'https://doi.org/10.5067/MODIS/MOD10A2.061',
    'comment': 'Cloud, darkness and no-decision observations are gap-filled bidirectionally: a gap is '
               'labelled snow only where the preceding and following clear observations both show snow '
               '(after Wrzesien et al. 2019). Tiles in the extreme polar rows additionally receive a '
               'polar-night correction so winter darkness is not misread as snow-free. The fill value '
               '-32768 marks pixels with no detected snow period: ocean, permanently snow-free land, and '
               'pixels with too few observations in the water year.',
    'references': 'Cloud gap-filling method: Wrzesien, M. L., Pavelsky, T. M., Durand, M. T., Dozier, J., '
                  '& Lundquist, J. D. (2019). Characterizing biases in mountain snow accumulation from '
                  'global data sets. Water Resources Research, 55(11), 9873-9891. '
                  'https://doi.org/10.1029/2019WR025350',
    'creator_name': 'Eric Gagliano',
    'creator_email': 'egagli@uw.edu',
    'creator_institution': 'University of Washington',
    'creator_url': 'https://orcid.org/0000-0002-4362-6260',
    'institution': 'University of Washington',
    'license': 'CC-BY-4.0',
    'source_repository': 'https://github.com/egagli/MODIS_snow_phenology',
    'config_version': config.VERSION,
    'icechunk_snapshot_id': str(snapshot_id),
    'date_created': created,
    'history': f'{created}: exported from Icechunk snapshot {snapshot_id} to plain Zarr v3 '
               'via notebooks/download_dataset.ipynb',
    # WY2015 opens Oct 1 2014 (NH); WY2025 closes Mar 31 2026 (SH)
    'time_coverage_start': f'{config.WY_START - 1}-10-01',
    'time_coverage_end': f'{config.WY_END + 1}-03-31',
    'time_coverage_resolution': 'P1Y',
    'geospatial_lat_min': -90.0,
    'geospatial_lat_max': 90.0,
    'geospatial_lon_min': -180.0,
    'geospatial_lon_max': 180.0,
    'spatial_resolution': '463.313 m (MODIS sinusoidal 500 m grid)',
}
if ZENODO_DOI:
    root_attrs['doi'] = ZENODO_DOI

long_names = {
    'SAD_DOWY': 'snow appearance date: first day of the longest continuous snow-covered period',
    'SDD_DOWY': 'snow disappearance date: first snow-free day following the longest continuous '
                'snow-covered period',
    'max_consec_snow_days': 'length of the longest continuous snow-covered period',
}


def patch_attributes(path, updates):
    meta = json.loads(path.read_text())
    meta.setdefault('attributes', {}).update(updates)
    path.write_text(json.dumps(meta, indent=2))


patch_attributes(STORE_PATH / 'zarr.json', root_attrs)
for var, long_name in long_names.items():
    patch_attributes(STORE_PATH / var / 'zarr.json', {'long_name': long_name})

print(json.dumps(json.loads((STORE_PATH / 'zarr.json').read_text())['attributes'], indent=2))

{
  "title": "Global MODIS snow phenology (snow appearance date, snow disappearance date, maximum consecutive snow days), water years 2015-2025",
  "description": "Snow appearance date (SAD), snow disappearance date (SDD), and maximum consecutive snow days derived from MODIS MOD10A2 8-day maximum snow extent product. Cloud filling follows Wrzesien et al. 2019.",
  "source": "MODIS/Terra Snow Cover 8-Day L3 Global 500m SIN Grid, Version 61 (MOD10A2.061), NASA National Snow and Ice Data Center DAAC",
  "Conventions": "CF-1.8, ACDD-1.3",
  "summary": "Annual global maps of three snow phenology metrics at 500 m resolution, derived from the MODIS/Terra MOD10A2 Version 61 8-day maximum snow extent product for water years 2015-2025. Snow appearance date (SAD_DOWY) and snow disappearance date (SDD_DOWY) bracket the longest continuous snow-covered period of each water year, whose length is max_consec_snow_days. Dates are given as day of water year (DOWY), where DOWY 1 is Oct 1 in the northern h

## 3. Verify

In [5]:
# Every source key landed on disk
local_keys = {p.relative_to(STORE_PATH).as_posix() for p in STORE_PATH.rglob('*') if p.is_file()}
missing = set(src_keys) - local_keys
print(f'source keys: {len(src_keys):,} | local keys: {len(local_keys):,} | missing: {len(missing)}')
assert not missing, sorted(missing)[:10]

# Array metadata matches (attribute patches aside)
ds_out = xr.open_zarr(STORE_PATH, zarr_format=3, consolidated=False, decode_coords='all')
for var in ['SAD_DOWY', 'SDD_DOWY', 'max_consec_snow_days']:
    a, b = ds_out[var], ds_src[var]
    assert a.shape == b.shape and a.dtype == b.dtype, var
    assert a.encoding['chunks'] == b.encoding['chunks'], var
for coord in ['water_year', 'y', 'x']:
    assert np.array_equal(ds_out[coord].values, ds_src[coord].values), coord
print(f'metadata + coordinates match | CRS: {ds_out.rio.crs.to_string()}')

source keys: 9,979 | local keys: 9,979 | missing: 0
metadata + coordinates match | CRS: PROJCS["unnamed",GEOGCS["Unknown datum based upon the custom spheroid",DATUM["Not specified (based on custom spheroid)",SPHEROID["Custom spheroid",6371007.181,0]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Sinusoidal"],PARAMETER["longitude_of_center",0],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["Meter",1],AXIS["Easting",EAST],AXIS["Northing",NORTH]]


In [6]:
# Spot-check raw int16 blocks against the source: random (variable, water year, tile) shards
TILE_PX = config.SHARD_SHAPE[1]
N_CHECKS = 12

raw = dict(zarr_format=3, consolidated=False, mask_and_scale=False)
out_raw = xr.open_zarr(STORE_PATH, **raw)
src_raw = xr.open_zarr(src, **raw)

tiles = config.get_process_tiles()
rng = np.random.default_rng(0)
picks = [
    (
        rng.choice(['SAD_DOWY', 'SDD_DOWY', 'max_consec_snow_days']),
        int(rng.choice(config.years)),
        tiles.iloc[int(rng.integers(len(tiles)))],
    )
    for _ in range(N_CHECKS)
]

for var, wy, tile in tqdm(picks, desc='spot-checking shards'):
    sel = dict(
        water_year=wy,
        y=slice(tile['v'] * TILE_PX, (tile['v'] + 1) * TILE_PX),
        x=slice(tile['h'] * TILE_PX, (tile['h'] + 1) * TILE_PX),
    )
    got = out_raw[var].sel(water_year=wy).isel(y=sel['y'], x=sel['x']).values
    want = src_raw[var].sel(water_year=wy).isel(y=sel['y'], x=sel['x']).values
    assert np.array_equal(got, want), (var, wy, tile['tile'])
    valid = int((got != -32768).sum())
    print(f'  {var:22s} WY{wy} {tile["tile"]}  identical, {valid:>9,d} valid px')

print(f'\nall {N_CHECKS} shards bit-identical to the source store')

  max_consec_snow_days   WY2022 h30v08  identical,    77,606 valid px


  SAD_DOWY               WY2018 h18v16  identical, 5,426,736 valid px


  SAD_DOWY               WY2015 h28v12  identical,   299,277 valid px


  max_consec_snow_days   WY2022 h15v02  identical, 2,206,211 valid px


  SDD_DOWY               WY2021 h18v01  identical,   385,545 valid px


...

  SDD_DOWY               WY2025 h03v10  identical,     3,448 valid px
  max_consec_snow_days   WY2022 h14v17  identical,         0 valid px


  SDD_DOWY               WY2024 h16v07  identical,    57,824 valid px


  SAD_DOWY               WY2023 h22v05  identical, 3,383,794 valid px


  max_consec_snow_days   WY2016 h14v14  identical,   199,308 valid px


  max_consec_snow_days   WY2015 h10v07  identical,    28,105 valid px

all 12 shards bit-identical to the source store


## 4. Archive

In [7]:
# ZIP_STORED: chunks are already zstd-compressed, and stored entries keep the zip
# directly readable by zarr.storage.ZipStore without extraction.
files = sorted(p for p in STORE_PATH.rglob('*') if p.is_file())
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_STORED, allowZip64=True) as zf:
    for p in tqdm(files, desc='archiving', unit='file'):
        zf.write(p, p.relative_to(STORE_PATH).as_posix())

zip_bytes = ZIP_PATH.stat().st_size
sha256 = hashlib.sha256()
with open(ZIP_PATH, 'rb') as f:
    for block in iter(lambda: f.read(16 << 20), b''):
        sha256.update(block)
sha256 = sha256.hexdigest()
print(f'{ZIP_PATH.name}: {zip_bytes / 1e9:.2f} GB')
print(f'sha256: {sha256}')

modis_snow_phenology_v1.zarr.zip: 3.61 GB
sha256: 9c988aa94d0bebdf6d05e533d8fb47e7675017ca655c16e195a4b3ec833171d3


In [8]:
# Read straight out of the archive, exactly as a user would
ds_zip = xr.open_zarr(zarr.storage.ZipStore(ZIP_PATH, mode='r'), zarr_format=3,
                      consolidated=False, decode_coords='all')
check = ds_zip['max_consec_snow_days'].sel(water_year=2020).rio.clip_box(
    minx=-121.95, miny=46.7, maxx=-121.45, maxy=46.95, crs='EPSG:4326')
print(f'Mount Rainier WY2020: {check.shape} px, '
      f'{float(check.min()):.0f}-{float(check.max()):.0f} consecutive snow days')
ds_zip

Mount Rainier WY2020: (61, 176) px, 5-366 consecutive snow days


<xarray.Dataset> Size: 493GB
Dimensions:               (water_year: 11, y: 43200, x: 86400)
Coordinates:
  * water_year            (water_year) int64 88B 2015 2016 2017 ... 2024 2025
  * y                     (y) float64 346kB 1.001e+07 1.001e+07 ... -1.001e+07
  * x                     (x) float64 691kB -2.001e+07 -2.001e+07 ... 2.001e+07
    spatial_ref           int64 8B ...
Data variables:
    max_consec_snow_days  (water_year, y, x) float32 164GB dask.array<chunksize=(1, 600, 600), meta=np.ndarray>
    SAD_DOWY              (water_year, y, x) float32 164GB dask.array<chunksize=(1, 600, 600), meta=np.ndarray>
    SDD_DOWY              (water_year, y, x) float32 164GB dask.array<chunksize=(1, 600, 600), meta=np.ndarray>
Attributes: (12/31)
    title:                     Global MODIS snow phenology (snow appearance d...
    description:               Snow appearance date (SAD), snow disappearance...
    source:                    MODIS/Terra Snow Cover 8-Day L3 Global 500m SI...
    Conventions:               CF-1.8, ACDD-1.3
    summary:                   Annual global maps of three snow phenology met...
    keywords:                  snow, snow cover, snow phenology, snow appeara...
    ...                        ...
    geospatial_lat_min:        -90.0
    geospatial_lat_max:        90.0
    geospatial_lon_min:        -180.0
    geospatial_lon_max:        180.0
    spatial_resolution:        463.313 m (MODIS sinusoidal 500 m grid)
    doi:                       10.5281/zenodo.21783366

In [9]:
# Facts to paste into the Zenodo record
print(f"""
file            {ZIP_PATH.name}
size            {zip_bytes / 1e9:.2f} GB ({zip_bytes:,} bytes)
sha256          {sha256}
unpacked        {store_bytes / 1e9:.2f} GB, {len(files):,} files
water years     {config.WY_START}-{config.WY_END}
shape           {dict(ds_out.sizes)}
dtype / fill    int16 / -32768
shards / chunks {config.SHARD_SHAPE} / {config.INNER_CHUNK_SHAPE}
crs             {ds_out.rio.crs.to_string()}
snapshot        {snapshot_id}
exported        {created}
""")


file            modis_snow_phenology_v1.zarr.zip
size            3.61 GB (3,605,775,330 bytes)
sha256          9c988aa94d0bebdf6d05e533d8fb47e7675017ca655c16e195a4b3ec833171d3
unpacked        3.60 GB, 9,979 files
water years     2015-2025
shape           {'water_year': 11, 'y': 43200, 'x': 86400}
dtype / fill    int16 / -32768
shards / chunks (1, 2400, 2400) / (1, 600, 600)
crs             PROJCS["unnamed",GEOGCS["Unknown datum based upon the custom spheroid",DATUM["Not specified (based on custom spheroid)",SPHEROID["Custom spheroid",6371007.181,0]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Sinusoidal"],PARAMETER["longitude_of_center",0],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["Meter",1],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
snapshot        M64XYVMFDN7GN756C3AG
exported        2026-08-04T03:10:30Z

